# Weather EDA Visualization

Notebook vẽ biểu đồ khám phá dữ liệu thời tiết Hà Nội.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/processed/Hanoi_final.csv").exists() and (PROJECT_ROOT.parent / "data/processed/Hanoi_final.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data/processed/Hanoi_final.csv"
df = pd.read_csv(DATA_PATH)
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)
df.head()

## 1. Time series nhiệt độ

In [ ]:
daily_temperature = df.set_index("time")["temperature_2m"].resample("D").mean()

plt.figure(figsize=(14, 5))
daily_temperature.plot(linewidth=1)
plt.title("Daily mean temperature in Hanoi")
plt.xlabel("Time")
plt.ylabel("Temperature (C)")
plt.tight_layout()

## 2. Phân phối các biến chính

In [ ]:
features = ["temperature_2m", "relative_humidity_2m", "rain", "pressure_msl", "wind_speed_10m", "cloud_cover"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, feature in zip(axes.ravel(), features):
    sns.histplot(df[feature], bins=40, kde=True, ax=ax)
    ax.set_title(feature)

plt.tight_layout()

## 3. Seasonality theo giờ và tháng

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.lineplot(data=df, x="hour", y="temperature_2m", errorbar=None, marker="o", ax=axes[0])
axes[0].set_title("Average temperature by hour")

sns.lineplot(data=df, x="month", y="temperature_2m", errorbar=None, marker="o", ax=axes[1])
axes[1].set_title("Average temperature by month")

plt.tight_layout()

## 4. Rainfall theo tháng

In [ ]:
monthly_rain = df.groupby("month", as_index=False)["rain"].sum()

plt.figure(figsize=(10, 5))
sns.barplot(data=monthly_rain, x="month", y="rain")
plt.title("Total rainfall by month")
plt.xlabel("Month")
plt.ylabel("Rain")
plt.tight_layout()

## 5. Heatmap tương quan

In [ ]:
corr_features = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "rain",
    "pressure_msl",
    "cloud_cover",
    "wind_speed_10m",
    "vapour_pressure_deficit",
]

plt.figure(figsize=(10, 8))
sns.heatmap(df[corr_features].corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation heatmap")
plt.tight_layout()

## 6. Quan hệ nhiệt độ với độ ẩm, mưa và mây

In [ ]:
sample_df = df.sample(min(8000, len(df)), random_state=42)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(data=sample_df, x="relative_humidity_2m", y="temperature_2m", alpha=0.35, ax=axes[0])
axes[0].set_title("Temperature vs humidity")

sns.scatterplot(data=sample_df, x="rain", y="temperature_2m", alpha=0.35, ax=axes[1])
axes[1].set_title("Temperature vs rain")

sns.scatterplot(data=sample_df, x="cloud_cover", y="temperature_2m", alpha=0.35, ax=axes[2])
axes[2].set_title("Temperature vs cloud cover")

plt.tight_layout()